In [ ]:
!pip -q install openai requests beautifulsoup4

In [ ]:
import json
import requests
from bs4 import BeautifulSoup
from dataclasses import dataclass, field
from typing import Any, Dict, List
from openai import OpenAI
import os,getpass
# Option 1: paste your key directly
# client = OpenAI(api_key="YOUR_API_KEY")

# Option 2: Colab secrets
from google.colab import userdata
api_key = getpass.getpass("Enter OpenAI API key: ")
client = OpenAI(api_key=api_key)

Enter OpenAI API key: ··········


In [ ]:
@dataclass
class AgentState:
    memory: List[str] = field(default_factory=list)
    conversation: List[Dict[str, str]] = field(default_factory=list)
    scratchpad: List[str] = field(default_factory=list)

In [ ]:
def fetch_web_content(url: str, max_chars: int = 6000) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    resp = requests.get(url, headers=headers, timeout=20)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")

    # remove noisy tags
    for tag in soup(["script", "style", "noscript"]):
        tag.extract()

    text = soup.get_text(separator=" ", strip=True)
    text = " ".join(text.split())

    if not text:
        return "No readable text found on the page."

    return text[:max_chars]

In [ ]:
TOOLS = [
    {
        "type": "function",
        "name": "fetch_web_content",
        "description": "Fetch readable text from a public webpage URL for later summarization.",
        "parameters": {
            "type": "object",
            "properties": {
                "url": {
                    "type": "string",
                    "description": "The public webpage URL to fetch"
                }
            },
            "required": ["url"],
            "additionalProperties": False
        }
    }
]

In [ ]:
def run_tool(tool_name: str, args: Dict[str, Any]) -> str:
    try:
        if tool_name == "fetch_web_content":
            return fetch_web_content(args["url"])
        return f"Unknown tool: {tool_name}"
    except Exception as e:
        return f"Tool error: {str(e)}"

In [ ]:
MODEL = "gpt-5.4-mini"

SYSTEM_PROMPT = """
You are a simple web-summary agent.

Rules:
1. If the user asks to summarize or inspect a webpage, call fetch_web_content.
2. Use memory when relevant.
3. Save short useful facts about the user into memory only if explicitly asked.
4. Be concise and accurate.
"""

def extract_text(response) -> str:
    if getattr(response, "output_text", None):
        return response.output_text

    parts = []
    for item in getattr(response, "output", []):
        if getattr(item, "type", None) == "message":
            for content in getattr(item, "content", []):
                if getattr(content, "type", None) == "output_text":
                    parts.append(content.text)
    return "\n".join(parts).strip()

def extract_function_calls(response):
    calls = []
    for item in getattr(response, "output", []):
        if getattr(item, "type", None) == "function_call":
            calls.append(item)
    return calls

def run_agent(state: AgentState, user_input: str, max_steps: int = 10) -> str:
    state.conversation.append({"role": "user", "content": user_input})

    memory_block = "\n".join(f"- {m}" for m in state.memory) if state.memory else "No memory yet."

    response = client.responses.create(
        model=MODEL,
        reasoning={"effort": "low"},
        tools=TOOLS,
        input=[
            {
                "role": "system",
                "content": f"{SYSTEM_PROMPT}\n\nMemory:\n{memory_block}"
            },
            *state.conversation
        ],
    )

    for step in range(max_steps):
        function_calls = extract_function_calls(response)

        if not function_calls:
            answer = extract_text(response)
            state.conversation.append({"role": "assistant", "content": answer})
            return answer

        tool_outputs = []

        for fc in function_calls:
            args = json.loads(fc.arguments)
            result = run_tool(fc.name, args)

            state.scratchpad.append(
                f"step={step+1} tool={fc.name} args={args} result_preview={result[:120]}"
            )

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": fc.call_id,
                "output": result
            })

        response = client.responses.create(
            model=MODEL,
            reasoning={"effort": "low"},
            tools=TOOLS,
            previous_response_id=response.id,
            input=tool_outputs
        )

    return "Stopped: max steps reached."

In [ ]:
state = AgentState()
url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
prompt = f"Fetch and summarize this page in 10 bullet points: {url}"

result = run_agent(state, prompt)
print(result)

- Artificial intelligence (AI) refers to computational systems that perform tasks typically associated with human intelligence.
- Common AI abilities include learning, reasoning, problem-solving, perception, and decision-making.
- The field covers both narrow AI systems and the broader goal of artificial general intelligence (AGI).
- AI methods include symbolic approaches, machine learning, deep learning, Bayesian networks, and evolutionary algorithms.
- Core AI goals include knowledge representation, planning, natural language processing, robotics, and general game playing.
- AI is used in many applications, including healthcare, finance, software development, translation, gaming, and generative AI.
- Modern AI has been strongly shaped by machine learning and especially deep learning.
- The article highlights important ethical concerns such as bias, privacy, misinformation, transparency, and job displacement.
- There are broader debates about AI safety, alignment, consciousness, right